# FAERS Signal Workbench: from reports to screening flags

This notebook explains what the Stage 3 signal table measures, how its counts and statistics are constructed, and what the results can (and cannot) tell us. It uses a small invented example first, then optionally inspects the local DuckDB output.

**Important:** FAERS contains reports of suspected events, not proof that a medicine caused an event. A signal is a reason to investigate, not a causal conclusion. The worked example below is illustrative and is not real FAERS data.

## 1. Why do disproportionality screening?

A large number of reports for a reaction does not by itself mean that one drug is unusually associated with it: widely used or heavily reported drugs will appear with many reactions. Disproportionality screening compares the share of reports mentioning a reaction for one suspect drug with that share among reports for other suspect drugs.

This is useful for **prioritizing review** across many drug–reaction pairs. It is not an incidence rate: FAERS does not provide the number of people exposed to each drug, and the number of reports is affected by use, publicity, reporting practices, and time delays. The comparator in this build is all other cases with a primary-suspect ingredient, not a matched or randomized control group.

## 2. One pair, one 2×2 table

Choose a drug X, reaction Y, and analysis window. Count each selected case once. The pair is X and Y; the row is whether the case's primary-suspect ingredient is X, and the column is whether the case mentions Y.

| | Mentions Y | Does not mention Y | Total |
|---|---:|---:|---:|
| Primary-suspect drug is X | **a** | **b** | a+b |
| Primary-suspect drug is another ingredient | **c** | **d** | c+d |
| Total | a+c | b+d | **N** |

In `signal_pairs`, `cases` = a, `drug_cases` = a+b, `reaction_cases` = a+c, and `total_cases` = N. Therefore b = `drug_cases - cases`, c = `reaction_cases - cases`, and d = `total_cases - drug_cases - reaction_cases + cases`. These are **pair-specific counts**: do not sum rows over different reactions and treat that as one contingency table, because cases can mention multiple reactions. Analyze one selected drug/reaction/window at a time.

## 3. Worked example: PRR, Yates chi-square, and uncertainty

Suppose the invented data have 50 reports mentioning Y among 1,000 reports for X, versus 500 reports mentioning Y among 100,000 reports for other drugs. The other-drug group therefore has 500 reports without Y. We can calculate all four cells and the screening statistics directly.

In [ ]:
from math import exp, log, sqrt

# Invented illustration only
a = 50
drug_cases = 1_000
reaction_cases = 550  # 50 for X + 500 for the other drugs
total_cases = 101_000

b = drug_cases - a
c = reaction_cases - a
d = total_cases - drug_cases - reaction_cases + a
N = a + b + c + d

prr = (a / (a + b)) / (c / (c + d))
yates_gap = max(0, abs(a * d - b * c) - N / 2)
chi_square = N * yates_gap**2 / ((a + b) * (c + d) * (a + c) * (b + d))
se_log_prr = sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
prr_ci_low = exp(log(prr) - 1.96 * se_log_prr)
prr_ci_high = exp(log(prr) + 1.96 * se_log_prr)

print(f"a={a:,}, b={b:,}, c={c:,}, d={d:,}, N={N:,}")
print(f"Y share for X:       {a/(a+b):.1%}")
print(f"Y share for others:  {c/(c+d):.1%}")
print(f"PRR:                 {prr:.2f}")
print(f"Yates chi-square:    {chi_square:.1f}")
print(f"Approx. 95% PRR CI:  {prr_ci_low:.2f} to {prr_ci_high:.2f}")

assert (a, b, c, d, N) == (50, 950, 500, 99_500, 101_000)
assert abs(prr - 10.0) < 1e-12
assert abs(chi_square - 361.9) < 0.5

### What the statistics mean

- **PRR (proportional reporting ratio)** compares `a/(a+b)` with `c/(c+d)`. Here it is 10: the reaction's share of X's reports is ten times its share among other drugs' reports. It is a ratio of reporting proportions, not a risk ratio from exposed and unexposed people.
- **Yates-corrected chi-square** measures departure from independence in this 2×2 table. The usual one-degree-of-freedom heuristic `chi-square ≥ 4` is roughly associated with p < 0.05, but statistical significance does not establish causality or clinical importance. The code uses `max(0, abs(ad-bc)-N/2)` before squaring; this standard continuity correction prevents a positive statistic when the correction exceeds the observed difference.
- **Approximate 95% PRR confidence interval** is calculated on the log scale using `sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))`. If its lower bound exceeds 1, the ratio is more consistently above 1 under this approximation. It still does not correct reporting bias, confounding, or selection effects.

The configured default flag is a **screening rule**: at least 3 cases, PRR ≥ 2, and Yates chi-square ≥ 4. Each cutoff is a prioritization choice, not a regulatory verdict. Many pairs are tested, so some will pass by chance; review patterns, evidence, and context rather than treating a flag as a discovery.

## 4. How Stage 3 builds the analysis table

The source clean tables have different grains. `clean_case` has one selected report per case; `clean_reaction` has one row per report and reaction term. Stage 3 joins these on `primaryid`, filters to `ps_status = 'PS'` with a non-null `ps_ingredient_std`, and groups by suspect ingredient and reaction key. Because reactions have already been collapsed to one row per report-term and the case table has one row per selected report, a case contributes once to a given pair.

For each pair the builder separately totals drug cases, reaction cases, and the eligible population so it can derive all four cells. It keeps pairs with at least three cases in each displayed window. The table contains 469,906 rows in the local build: 363,016 for **All quarters** and 106,890 for **Last 2 quarters**. The all-quarter denominator is 2,857,989 primary-suspect cases; the latest-two-quarter denominator is 673,198 cases (26Q1 and 26Q2). These are counts of reports, not patients or drug exposures.

### Time windows and emerging pairs

`Last 2 quarters` follows the `is_provisional` flag in the clean case table; in this snapshot that means 26Q1 and 26Q2. Newer data are provisional because recent reports can be delayed or incomplete. `Earlier quarters` is an internal comparison window, not a displayed signal-pair window. `is_emerging` is true only on a recent-window row when the recent pair meets the default flag and the same ingredient/reaction did not meet it in earlier quarters. It means “newly passes this rule in this extract,” not “new adverse effect” or “caused by the drug.”

### Serious outcomes, duplicate sensitivity, and administrative terms

`serious_cases` counts cases with any serious outcome code; `death_cases` counts cases with a death outcome. These are counts among cases reporting the pair, not estimates of severity caused by the drug.

The `*_dr` fields recompute pair cells and statistics after excluding cases marked as later members of **candidate** duplicate groups. `flagged_dr_default` applies the same default thresholds to those counts; `robust_to_duplicate_reduction` means both the original and candidate-reduced versions pass. Duplicate candidates are not confirmed duplicates, so this is a sensitivity analysis, not a replacement truth set.

`is_administrative` marks a configurable list of product-use / non-clinical reaction terms. The rows are retained so the list can be reviewed; dashboards may hide them by default but should not erase them from the source table.

## 5. What can mislead us?

Use flags to guide investigation, not to make a causal claim. Important limitations include:

- **No exposure denominator:** more reports do not mean more risk; FAERS does not tell us how many people took each medicine.
- **Reporting bias and publicity:** media attention, litigation, manufacturer programs, and changes in reporting can change counts.
- **Confounding and disease symptoms:** the underlying disease, age, co-medications, and treatment indication can explain an event.
- **Duplicates and follow-ups:** candidate duplicate rules are imperfect; recent reports and case histories can change as new information arrives.
- **Comparator choice:** “all other suspect drugs” is a broad, potentially dissimilar population dominated by frequently reported products.
- **Small counts and many tests:** a cutoff does not eliminate chance findings; estimates from small cells can be unstable. A PRR may be undefined when the comparator has zero reports of the reaction.
- **Terms and product identity:** administrative terms can dominate lists; combination ingredients remain a combined label and are not split into components.

A high PRR can result from a small comparator share, coding practices, or unusually low comparator reporting. Check the counts, confidence interval, time pattern, seriousness, duplicate sensitivity, and external clinical evidence before drawing conclusions.

## 6. Using the counts in Tableau

For one filtered `pair_id` and one `window`, Tableau can calculate:

```text
[a] = SUM([cases])
[b] = SUM([drug_cases]) - [a]
[c] = SUM([reaction_cases]) - [a]
[d] = MAX([total_cases]) - [a] - [b] - [c]
[PRR] = ([a] / ([a] + [b])) / ([c] / ([c] + [d]))
```

Do not total these values across different reaction rows and then interpret the result as one 2×2 test: one case may mention many reactions, so rows overlap. Use `pair_id` (ingredient + canonical reaction key) and `window` as the analysis grain. The precomputed `prr` and `chi_square` are available for cross-checking Tableau calculations. Guard against zero denominators and hide administrative terms with a user-controllable filter rather than deleting them.

## 7. Optional: inspect the local DuckDB result (read-only)

This section looks for the verified local database. The database is large and intentionally excluded from Git, so a fresh clone may not have it. If needed, edit `DB_PATH` below. The connection is read-only and the queries select aggregate pair rows only—no case-level records are displayed.

In [ ]:
from pathlib import Path

here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (here, *here.parents) if (p / ".git").exists()), here)
DB_PATH = PROJECT_ROOT / "outputs" / "stage2_clean_20261029_verified" / "faers_clean.duckdb"

if DB_PATH.is_file():
    import duckdb
    con = duckdb.connect(str(DB_PATH), read_only=True)
    print(f"Connected read-only: {DB_PATH}")
else:
    con = None
    print(f"Local DuckDB not found at {DB_PATH}. The synthetic example above is complete; edit DB_PATH to inspect another local database.")

In [ ]:
if con is not None:
    tables = con.execute("""
        SELECT table_name FROM information_schema.tables
        WHERE table_schema = 'main' AND table_name IN
              ('clean_case', 'clean_reaction', 'signal_pairs', 'signal_meta', 'signal_admin_terms')
        ORDER BY table_name
    """).fetchall()
    print("Relevant tables:", [row[0] for row in tables])
    print("signal_pairs columns:")
    for column in con.execute("DESCRIBE signal_pairs").fetchall():
        print(f"  {column[0]}: {column[1]}")
    print("\nBuild metadata:")
    for row in con.execute("SELECT key, value FROM signal_meta ORDER BY key").fetchall():
        print(f"  {row[0]}: {row[1]}")

In [ ]:
if con is not None:
    window_summary = con.execute("""
        SELECT "window", count(*) AS pairs,
               count(*) FILTER (WHERE flagged_default) AS flagged,
               count(*) FILTER (WHERE robust_to_duplicate_reduction) AS robust_after_candidate_reduction,
               count(*) FILTER (WHERE is_emerging) AS emerging,
               count(*) FILTER (WHERE is_administrative) AS administrative
        FROM signal_pairs GROUP BY 1 ORDER BY 1
    """).fetchall()
    for row in window_summary:
        print(row)

In [ ]:
if con is not None:
    # Pick a readable example; change the window or sort to explore other pairs.
    sample = con.execute("""
        SELECT "window", ps_ingredient_std, pt, pt_key, cases, drug_cases,
               reaction_cases, total_cases, prr, chi_square, prr_ci_low,
               serious_cases, death_cases, flagged_default,
               flagged_dr_default, robust_to_duplicate_reduction,
               is_emerging, is_administrative
        FROM signal_pairs
        WHERE "window" = 'All quarters' AND flagged_default AND NOT is_administrative
        ORDER BY cases DESC LIMIT 1
    """).fetchone()
    if sample is None:
        print("No matching sample pair found.")
    else:
        (window, drug, reaction, pt_key, a, drug_cases, reaction_cases, total_cases,
         stored_prr, stored_chi, stored_ci_low, serious, deaths, flagged,
         flagged_dr, robust, emerging, administrative) = sample
        b = drug_cases - a
        c = reaction_cases - a
        d = total_cases - drug_cases - reaction_cases + a
        n = a + b + c + d
        prr = (a / (a + b)) / (c / (c + d)) if c > 0 else None
        yates_gap = max(0, abs(a * d - b * c) - n / 2)
        chi_square = n * yates_gap**2 / ((a+b) * (c+d) * (a+c) * (b+d))
        se_log_prr = sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d)) if c > 0 else None
        ci_low = exp(log(prr) - 1.96 * se_log_prr) if prr is not None else None
        print(f"{window}: {drug} / {reaction} ({pt_key})")
        print(f"cells: a={a:,}, b={b:,}, c={c:,}, d={d:,}, N={n:,}")
        print(f"PRR: recomputed={prr:.4f}, stored={stored_prr:.4f}")
        print(f"Yates chi-square: recomputed={chi_square:.2f}, stored={stored_chi:.2f}")
        print(f"PRR lower 95% bound: recomputed={ci_low:.4f}, stored={stored_ci_low:.4f}")
        print(f"serious={serious:,}, deaths={deaths:,}, flagged={flagged}, duplicate-robust={robust}, emerging={emerging}, administrative={administrative}")
        assert abs(prr - stored_prr) < 1e-10 * max(1, abs(prr))
        assert abs(chi_square - stored_chi) < 1e-10 * max(1, abs(chi_square))
        assert abs(ci_low - stored_ci_low) < 1e-10 * max(1, abs(ci_low))

## 8. Take-home points

1. The table makes a large FAERS scan practical by precomputing counts at the drug–reaction–window grain.
2. PRR describes relative reporting proportions; chi-square measures incompatibility with independence under a simple model; the interval describes approximate statistical uncertainty. None is a causal test.
3. Read the underlying counts and definitions before interpreting a large number. A threshold flag is a queue for expert review, not a conclusion.
4. Keep recent/provisional status, candidate-duplicate sensitivity, serious outcomes, administrative terms, and the comparator definition visible to dashboard users.

Close the optional connection when finished:

In [ ]:
if con is not None:
    con.close()
    print("Read-only DuckDB connection closed.")